# ForgeLM — KTO Binary Feedback Alignment

Align a model using simple thumbs-up/thumbs-down feedback — no paired preferences needed.

**Requirements:** GPU runtime required (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/kto_binary_feedback.ipynb)

In [ ]:
# Step 1: Install ForgeLM
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git bitsandbytes
!pip uninstall -y wandb -q
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 4: Create config
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "kto"
  kto_beta: 0.1
  output_dir: "./kto_checkpoints"
  num_train_epochs: 1
  per_device_train_batch_size: 4
  learning_rate: 5.0e-6

data:
  dataset_name_or_path: "kto_data.jsonl"
"""

with open("kto_config.yaml", "w") as f:
    f.write(config_yaml)
print("KTO config saved!")

In [ ]:
# Step 4: Create config
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "kto"
  kto_beta: 0.1
  output_dir: "./kto_checkpoints"
  num_train_epochs: 1
  per_device_train_batch_size: 2
  learning_rate: 5.0e-6

data:
  dataset_name_or_path: "kto_data.jsonl"
"""

with open("kto_config.yaml", "w") as f:
    f.write(config_yaml)
print("KTO config saved!")

In [ ]:
# Step 5: Validate
!forgelm --config kto_config.yaml --dry-run

In [ ]:
# Step 6: Train
!forgelm --config kto_config.yaml

In [ ]:
# Step 7: Verify output
import os

model_path = "./kto_checkpoints/final_model"
if not os.path.exists(model_path):
    print(f"Error: '{model_path}' not found. Check training output above.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"Error: adapter_config.json not found in '{model_path}'.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")